In [87]:
import os
import re
import json
import torch
import pandas as pd
import scanpy as sc
import math
import logging
import importlib
from pathlib import Path
import warnings
import numpy as np
import traceback
import scipy.sparse as sp
import random
from scipy.special import softmax
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import Tuple, Set, Optional, List, Iterable, Union, Dict, Callable, Any
from anndata import AnnData
import pybedtools
import argparse
import pickle
from sklearn.decomposition import TruncatedSVD

import sys
PROJECT_DIR = "/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER"
SRC_DIR = str(Path(PROJECT_DIR) / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from multiomic_transformer.utils.peaks import find_genes_near_peaks, format_peaks
import multiomic_transformer.utils.data_formatter as data_formatter

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

# Training Data Preparation and Caching

The model uses cached per-chromosome data to help speed up training. Here, we will go through the process of building the necessary tensors and reference files to train the gene prediction model.

## Set up the TrainingDataFormatter

The training data formatter ensures that the correct data files and output directories exist and helps to keep everything standardized.

In [88]:
importlib.reload(data_formatter)


<module 'multiomic_transformer.utils.data_formatter' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/data_formatter.py'>

In [89]:
# Path to the project directory (same as Git repository root)
project_dir = Path(PROJECT_DIR)

# Path to the training output directory. Used to store the preprocessing config
output_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments")

# Name of the dataset / experiment to run
dataset_name = "mESC_E7.5_rep1_hvg_filter_disp_0.6"

# Organism code for the dataset. Supports either "mm10" or "hg38"
organism_code = "mm10"

# List of samples in the training datset. 
# Each of these should have its own subdirectory in the processed data directory
sample_names = ["E7.5_rep1"]

# List of chromosomes. Used to split the data by chromsome for caching and training.
# Should be in the format "chr1", "chr2", etc. and should match the chromosome names in the processed data files.
chrom_list = [f"chr{i}" for i in range(1, 20)]

tdf = data_formatter.TrainingDataFormatter(
    project_dir=project_dir,
    dataset_name=dataset_name,
    organism_code=organism_code,
    sample_names=sample_names,
    chrom_list=chrom_list,
    output_dir=output_dir / dataset_name,
)

with open(tdf.output_dir / "file_paths.json", "r") as f:
    file_paths = json.load(f)
    
# print(json.dumps(file_paths, indent=2))

  - Found existing genome file: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/reference_genome/mm10/mm10.fa.gz
  - mm10.fa.gz is already BGZF; skipping transcode
  - Index already exists: mm10.fa.gz.fai, mm10.fa.gz.gzi
Genome ready: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/reference_genome/mm10/mm10.fa.gz
Found existing gene_info: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/genome_annotation/mm10
Found existing GTF: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/genome_annotation/mm10/Mus_musculus.GRCm39.115.gtf.gz
Map sizes: {'n_official': 143320, 'n_ens2sym': 77920, 'n_entrez2sym': 112254, 'n_alias2sym': 72569}


We can check that all of the required cached files exist for the experiment.

In [90]:
files_exist = tdf.check_cached_file_exist()

All required files are present.


## Sample-Level Data Preparation

We first load the pseudobulk data from the Muon QC and preprocssing pipeline. The data files are loaded and the peaks and genes are standardized.

In [91]:
sample_name = sample_names[0]

pseudobulk_rna_df = tdf.load_pseudobulk_rna_df(sample_name)
pseudobulk_atac_df = tdf.load_pseudobulk_atac_df(sample_name)

# Loading the pseudobulk data also creates the TF and TG name files and stores the TF and TG names as a list, 
# and creates the peak BED file and stores the peak locations as a dataframe.
genes = tdf.genes
tfs = tdf.tfs
tgs = tdf.tgs

peaks = tdf.peaks
print(f"Number of genes: {len(genes):,}")
print(f"  - TFs: {len(tfs):,} (First 3: {tfs[:3]})")
print(f"  - TGs: {len(tgs):,} (First 3: {tgs[:3]})")
print(f"Number of peaks: {len(peaks):,} (First 3: {peaks[:3]})")

peak_bed_df = tdf.peak_locs_df
display(peak_bed_df.head())

  - Formatting 50159 peaks


Number of genes: 2,603
  - TFs: 256 (First 3: ['AHR', 'AHRR', 'ALX1'])
  - TGs: 2,660 (First 3: ['0610009E02RIK', '1110046J04RIK', '1500015A07RIK'])
Number of peaks: 50,159 (First 3: ['chr1:3142536-3143136', 'chr1:3553099-3553699', 'chr1:3584996-3585596'])


,chrom,start,end,strand,peak_id
0,chr1,3142536,3143136,.,chr1:3142536-3143136
1,chr1,3553099,3553699,.,chr1:3553099-3553699
2,chr1,3584996,3585596,.,chr1:3584996-3585596
3,chr1,3619552,3620152,.,chr1:3619552-3620152
4,chr1,3741734,3742334,.,chr1:3741734-3742334


### Calculate peak to gene distance

Next, we will use the peak bed file created when loading the peak pseudobulk dataset to calculate the distance between each peak and each gene within 1 Mb. This will create a bias tensor to help the model prioritize genes that are closer to peaks when training the peak-TG cross-attention portion of the model.

In [92]:
tdf.create_peak_to_tg_distance_file(
    sample_name=sample_name,
    max_peak_distance=1e6,
    distance_factor_scale=25000,
    force_recalculate=False,
    filter_to_nearest_gene=False,
)

Peak to TSS distance file already exists for sample E7.5_rep1 at /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/processed/mESC_E7.5_rep1_hvg_filter_disp_0.6/E7.5_rep1/peak_to_gene_dist.parquet. Skipping recalculation.


,peak_chr,peak_start,peak_end,peak_id,gene_chr,gene_start,gene_end,target_id,TSS_dist,TSS_dist_score
0,chr8,120397119,120397719,chr8:120397119-120397719,chr8,120397719,120397720,GM32352,0,1.000000e+00
1,chr5,53147843,53148443,chr5:53147843-53148443,chr5,53148443,53148444,5033403H07RIK,0,1.000000e+00
2,chr12,14333068,14333668,chr12:14333068-14333668,chr12,14333668,14333669,GM40840,0,1.000000e+00
3,chr6,122370156,122370756,chr6:122370156-122370756,chr6,122370756,122370757,GM68560,0,1.000000e+00
4,chr17,69153709,69154309,chr17:69153709-69154309,chr17,69154309,69154310,GM64628,0,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...
12317036,chr7,27071403,27072003,chr7:27071403-27072003,chr7,28072002,28072003,PLEKHG2,999999,4.248524e-18
12317037,chr8,113580323,113580923,chr8:113580323-113580923,chr8,112580923,112580924,CFDP1,1000000,4.248354e-18
12317038,chr2,168618841,168619441,chr2:168618841-168619441,chr2,167619441,167619442,9230111E07RIK,1000000,4.248354e-18
12317039,chr17,84389343,84389943,chr17:84389343-84389943,chr17,83389943,83389944,GM66646,1000000,4.248354e-18


## Chromosome-Specific Data Formatting


### Build Global TG Vocab



In [ ]:
tg_vocab = tdf.build_global_tg_vocab()

